<a href="https://colab.research.google.com/github/chitta-behera/Machine-Learning/blob/master/classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [2]:
news = [

    "India won the cricket match",
    "Virat Kohli scored a century",
    "New football season begins",

    "Stock market reached new high",
    "RBI increased interest rates",
    "Company profits increased",

    "New AI model released",
    "Python is popular for AI",
    "Cloud computing is growing",

    "Election results announced",
    "Government passed new law",
    "Prime minister addressed parliament"
]

In [3]:
labels = [

    0,0,0,

    1,1,1,

    2,2,2,

    3,3,3
]

In [4]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [5]:
news = [

    clean_text(article)

    for article in news
]

In [6]:
tokenized = []

for article in news:

    tokenized.append(

        article.split()

    )

print(tokenized)

[['india', 'won', 'the', 'cricket', 'match'], ['virat', 'kohli', 'scored', 'a', 'century'], ['new', 'football', 'season', 'begins'], ['stock', 'market', 'reached', 'new', 'high'], ['rbi', 'increased', 'interest', 'rates'], ['company', 'profits', 'increased'], ['new', 'ai', 'model', 'released'], ['python', 'is', 'popular', 'for', 'ai'], ['cloud', 'computing', 'is', 'growing'], ['election', 'results', 'announced'], ['government', 'passed', 'new', 'law'], ['prime', 'minister', 'addressed', 'parliament']]


In [7]:
vocab = {}

index = 1

for sentence in tokenized:

    for word in sentence:

        if word not in vocab:

            vocab[word] = index

            index += 1

print(vocab)

{'india': 1, 'won': 2, 'the': 3, 'cricket': 4, 'match': 5, 'virat': 6, 'kohli': 7, 'scored': 8, 'a': 9, 'century': 10, 'new': 11, 'football': 12, 'season': 13, 'begins': 14, 'stock': 15, 'market': 16, 'reached': 17, 'high': 18, 'rbi': 19, 'increased': 20, 'interest': 21, 'rates': 22, 'company': 23, 'profits': 24, 'ai': 25, 'model': 26, 'released': 27, 'python': 28, 'is': 29, 'popular': 30, 'for': 31, 'cloud': 32, 'computing': 33, 'growing': 34, 'election': 35, 'results': 36, 'announced': 37, 'government': 38, 'passed': 39, 'law': 40, 'prime': 41, 'minister': 42, 'addressed': 43, 'parliament': 44}


In [8]:
encoded = []

for sentence in tokenized:

    temp = []

    for word in sentence:

        temp.append(vocab[word])

    encoded.append(temp)

print(encoded)

[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14], [15, 16, 17, 11, 18], [19, 20, 21, 22], [23, 24, 20], [11, 25, 26, 27], [28, 29, 30, 31, 25], [32, 33, 29, 34], [35, 36, 37], [38, 39, 11, 40], [41, 42, 43, 44]]


In [9]:
max_len = max(

    len(sentence)

    for sentence in encoded
)

In [10]:
padded = []

for sentence in encoded:

    while len(sentence) < max_len:

        sentence.append(0)

    padded.append(sentence)

print(padded)

[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 0], [15, 16, 17, 11, 18], [19, 20, 21, 22, 0], [23, 24, 20, 0, 0], [11, 25, 26, 27, 0], [28, 29, 30, 31, 25], [32, 33, 29, 34, 0], [35, 36, 37, 0, 0], [38, 39, 11, 40, 0], [41, 42, 43, 44, 0]]


In [11]:
X = torch.LongTensor(padded)

y = torch.LongTensor(labels)

In [12]:
class NewsDataset(Dataset):

    def __init__(self,X,y):

        self.X = X

        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self,index):

        return self.X[index],self.y[index]

In [13]:
dataset = NewsDataset(X,y)

loader = DataLoader(

    dataset,

    batch_size=2,

    shuffle=True
)

In [16]:
class NewsClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(

            num_embeddings=len(vocab)+1,

            embedding_dim=16
        )

        self.fc = nn.Sequential(

            nn.Linear(16,32),

            nn.ReLU(),

            nn.Linear(32,16),

            nn.ReLU(),

            nn.Linear(16,4)
        )

    def forward(self,x):

        x = self.embedding(x)

        x = x.mean(dim=1)

        x = self.fc(x)

        return x

In [17]:
model = NewsClassifier()

In [18]:
criterion = nn.CrossEntropyLoss()

In [19]:
optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.01
)

In [20]:
epochs = 50

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        outputs = model(X_batch)

        loss = criterion(

            outputs,

            y_batch
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:

        print(

            f"Epoch {epoch} Loss {total_loss:.4f}"

        )

Epoch 0 Loss 8.4615
Epoch 10 Loss 1.1201
Epoch 20 Loss 0.0108
Epoch 30 Loss 0.0049
Epoch 40 Loss 0.0029


In [21]:
test = "AI is transforming software development"

In [22]:
test = clean_text(test)
words = test.split()
encoded = []

for word in words:

    encoded.append(

        vocab.get(word,0)

    )

In [23]:
while len(encoded) < max_len:

    encoded.append(0)

In [24]:
test = torch.LongTensor([encoded])

In [25]:
model.eval()

with torch.no_grad():

    output = model(test)

    prediction = torch.argmax(

        output,

        dim=1
    )

print(prediction.item())

2


In [26]:
classes = {

    0:"Sports",

    1:"Business",

    2:"Technology",

    3:"Politics"
}

print(

    classes[prediction.item()]

)

Technology
